[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/raya-lucaria/ia_o26/blob/main/course/7_juegos/_assets/01_decidir_en_el_arbol_t.ipynb)

# Notebook · Decidir en el árbol T

Cinco algoritmos en el mismo árbol de juguete: **minimax**,
**expectiminimax**, **alfa-beta**, **minimax con corte** y **profundización
iterativa**. El código es el Python de las páginas, con su número de línea
`# (n)`.

- Primero: **Entorno de ejecución → Ejecutar todas** (`Ctrl + F9`).
- La sección 1 es maquinaria: no hace falta leerla.
- Mueve el deslizador **paso** (o dale ▶): ves el árbol, la línea que corre
  y sus variables. Cada sección cierra con un **Pruébalo**.

## 1 · Preparación

Dos celdas plegadas: las reglas del árbol y el visor. Córrelas y sigue.

In [ ]:
# @title Reglas y árbol (no hace falta leerla)
from fractions import Fraction
from math import inf
import inspect
import re

# R (MAX) → I, C, D (MIN) por izq, centro, der;
# C → C1, C2 (MAX) por c1, c2. Los números son hojas.
ARBOL_T = ((3, 6), ((5, 2), (7, 8)), (2, 12))
ARBOL_T1 = ((3, 6), (2, 12))                 # T sin el centro
EVAL_T = {"I": 5, "C": 4, "D": 7, "C1": 6, "C2": 9}
JUGADAS_R = {3: ["izq", "centro", "der"], 2: ["izq", "der"]}
NOMBRE_R = {"izq": "I", "centro": "C", "der": "D"}


def armar(arbol, s="R", k=0, t=None):
    """Nombra los nodos: R; I, C, D; C1, C2; una hoja, «padre-lugar»."""
    t = t or dict(hijos={}, valor={}, prof={}, padre={})
    t["prof"][s] = k
    if not isinstance(arbol, tuple):
        t["valor"][s] = arbol
        return t
    t["hijos"][s] = {}
    for i, h in enumerate(arbol, 1):
        a = JUGADAS_R[len(arbol)][i - 1] if k == 0 else f"{s.lower()}{i}"
        sep = "" if isinstance(h, tuple) else "-"
        n = NOMBRE_R[a] if k == 0 else f"{s}{sep}{i}"
        t["hijos"][s][a], t["padre"][n] = n, s
        armar(h, n, k + 1, t)
    return t


def usar(arbol, azar=(), cien=False, al_reves=False):
    """Elige el árbol. azar: los nodos de azar. cien: la escala de la
    clase 3 (la hoja 3 es un final con U = 3/100). al_reves: cada A(s)
    en el orden inverso, sin cambiar nombres."""
    global JUEGO
    JUEGO = armar(arbol)
    JUEGO.update(azar=set(azar), cien=cien, al_reves=al_reves)


# --- Las reglas: S_F, U, Pl, A, T, Pr y EVAL -------------------------
def es_final(s):
    return s in JUEGO["valor"]

def utilidad(s):
    u = JUEGO["valor"][s]
    return Fraction(u, 100) if JUEGO["cien"] else u

def pl(s):
    if s in JUEGO["azar"]:
        return "AZAR"
    return ("MAX", "MIN")[JUEGO["prof"][s] % 2]

def acciones(s):
    a = list(JUEGO["hijos"][s])
    return a[::-1] if JUEGO["al_reves"] else a

def pr(s, a):
    return Fraction(1, len(JUEGO["hijos"][s]))

# Generar un nodo es llamar a transicion: la cuenta lo anota.
CUENTA = dict(generados=["R"], total=0, evaluados=set(), terminadas=0)

def transicion(s, a):
    hijo = JUEGO["hijos"][s][a]
    CUENTA["generados"].append(hijo)
    CUENTA["total"] += 1
    return hijo

def evaluar(s):                          # EVAL: una estimación de s
    CUENTA["evaluados"].add(s)
    return EVAL_T[s]


# --- paso: la foto de una fila de la traza ---------------------------
VISIBLES = ("s", "a", "v", "w", "alfa", "beta", "mejor_valor",
            "mejor_jugada", "jugada", "d")
EVENTO_R = {"2": "inicio", "5": "inicio", "2–3": "antes", "4–5": "raiz",
            "7–10": "raiz", "6": "fin", "11–12": "fin", "13": "entrega"}

def paso(linea, variables, evento=None, **extra):
    """Una fila: la línea de la página y las variables en ese instante."""
    x = {k: variables[k] for k in VISIBLES if k in variables}
    if evento is None and x["s"] == "R":
        evento = EVENTO_R[linea]
    evento = evento or ("regresa" if "–" in linea else "entra")
    if evento == "inicio" and linea == "5":   # otra búsqueda: la cuenta
        CUENTA.update(generados=["R"], evaluados=set(),  # vuelve a R
                      total=CUENTA["total"] + 1)
    if evento == "fin":
        CUENTA["terminadas"] += 1          # para el reloj de la sección 7
    return dict(evento=evento, linea=linea, vars=x, t=JUEGO, **extra,
                generados=list(CUENTA["generados"]),
                evaluados=set(CUENTA["evaluados"]))

def corte(linea, variables):
    """La fila del corte y una fila ✗ por cada hijo que ya no se genera."""
    s, a = variables["s"], variables["a"]
    tipo = "beta" if pl(s) == "MAX" else "alfa"
    yield paso(linea, variables, "corte " + tipo)
    for b in acciones(s)[acciones(s).index(a) + 1:]:
        yield paso("", {"s": s}, "podado", nodo=JUEGO["hijos"][s][b],
                   corta=tipo)


# --- correr: de fotos a la tabla de la página ------------------------
CARGA = ("mejor_jugada", "jugada", "d", "alfa")

def camino(t, n):                        # la pila: de R hasta n
    return camino(t, t["padre"][n]) + [n] if n in t["padre"] else [n]

def fila_de(n, p, arriba):
    x, ev, t = p["vars"], p["evento"], p["t"]
    s = x["s"]
    f = dict(n=n, evento=ev, linea=p["linea"], nodo=p.get("nodo", s),
             pila=camino(t, s), v=x.get("v"), w=None, hijo=None,
             alfa=x.get("alfa"), beta=x.get("beta"), corta=p.get("corta"),
             hechos=p["generados"], evaluados=p["evaluados"], t=t)
    if s == "R":                         # en R: α de la raíz y β = +∞
        arriba.update({k: x[k] for k in CARGA if k in x})
        f["alfa"] = arriba.get("alfa")
        f["beta"] = None if f["alfa"] is None else inf
        f["v"] = x.get("mejor_valor", f["alfa"])
    if ev in ("regresa", "raiz", "corte beta", "corte alfa"):
        f.update(w=x["w"], hijo=t["hijos"][s][x["a"]])
        f["corta"] = ev.split()[1] if ev.startswith("corte") else None
    if ev == "podado":
        f["w"] = t["valor"].get(f["nodo"])
    return dict(f, **{k: arriba.get(k) for k in CARGA[:3]})

def correr(algoritmo):
    """Corre el generador. Devuelve (filas, lo que devolvió)."""
    CUENTA.update(generados=["R"], total=0, evaluados=set(), terminadas=0)
    pasos = []
    while True:
        try:
            pasos.append(next(algoritmo))
        except StopIteration as fin:
            resultado = fin.value
            break
    arriba = {}
    filas = [fila_de(n, p, arriba) for n, p in enumerate(pasos, 1)]
    return filas, resultado


# --- el estado de cada nodo, fila por fila ---------------------------
def bajo(t, n):                          # n y sus descendientes
    return [n] + [m for h in t["hijos"].get(n, {}).values()
                  for m in bajo(t, h)]

def recordar(f, m):
    t = f["t"]
    if f["evento"] in ("antes", "inicio"):     # empieza una búsqueda
        m.update(v={}, ab={}, dev={}, podados=set(), cotas=set())
    if f["evento"] != "podado":
        m["v"][f["nodo"]] = f["v"]
        m["ab"][f["nodo"]] = (f["alfa"], f["beta"])
    if f["hijo"] is not None:
        m["dev"][f["hijo"]] = f["w"]
    if f["evento"].startswith("corte"):
        fuera = [h for h in t["hijos"][f["nodo"]].values()
                 if h not in f["hechos"]]
        m["podados"].update(y for h in fuera for y in bajo(t, h))
        m["cotas"].update([f["nodo"]] if fuera else [])

def estado_de(n, f, m):
    e = dict(estado="pormirar", v=None, alfa=None, beta=None, cota=False)
    abiertos = () if f["evento"] in ("fin", "entrega") else f["pila"]
    valor = f["t"]["valor"]
    if n in abiertos:
        a, b = m["ab"].get(n, (None, None))
        e.update(estado="pila", v=m["v"].get(n), alfa=a, beta=b)
    elif n in f["hechos"]:
        v = valor[n] if n in valor else m["dev"].get(n, m["v"].get(n))
        ev = "evaluado" if n in f["evaluados"] else "devuelto"
        e.update(estado=ev, v=v, cota=n in m["cotas"])
    elif n in m["podados"]:
        e["estado"] = "podado"
    return e

def fotos_de(filas):
    m = {}
    return [recordar(f, m) or {n: estado_de(n, f, m) for n in f["t"]["prof"]}
            for f in filas]


# --- imprimir ---------------------------------------------------------
def fmt(x):
    return "" if x is None else {inf: "+∞", -inf: "−∞"}.get(x, str(x))

def renglon(f):
    """# · línea · pila · (α, β) · v · w · ¿corta? · mejor_jugada"""
    if f["evento"] == "podado":
        return (f"{f['n']:>2} ✗ {f['nodo']} ({fmt(f['w'])}) no se genera:"
                f" corte {f['corta']}")
    ab = "" if f["alfa"] is None else f"({fmt(f['alfa'])}, {fmt(f['beta'])})"
    w = fmt(f["w"]) + (f" ({f['hijo']})" if f["hijo"] and "-" not in
                       f["hijo"] else "")
    return (f"{f['n']:>2} │ {f['linea']:<5} │ {'›'.join(f['pila']):<8} │ "
            f"{ab:<9} │ v {fmt(f['v']):<4} │ w {w:<8} │ "
            f"{f['corta'] or '':<4} │ {f['mejor_jugada'] or 'ninguna'}"
            f" │ gen {len(f['hechos'])}")

def tabla(filas):
    for f in filas:
        print(renglon(f))


# --- el código, con ▶ en las líneas del paso -------------------------
def rango(linea):                        # «12–15» → [12, 13, 14, 15]
    a, _, b = linea.partition("–")
    return list(range(int(a), int(b or a) + 1)) if a else []

def marcas(fn):
    return {int(n) for n in re.findall(r"# \((\d+)\)",
                                       inspect.getsource(fn))}

def elegir(f, fuentes):                  # la función que hace ese paso
    ls = set(rango(f["linea"]))
    return max(fuentes, key=lambda g: len(marcas(g) & ls))

def con_flechas(fuente, ls):
    """Sin las fotos; un (n) vale para su línea y las que le siguen,
    hasta otro (n) o hasta salir de su bloque."""
    actual, base = None, 0
    for r in fuente.splitlines():
        if re.search(r"\b(paso|corte)\(", r):
            continue
        ind = len(r) - len(r.lstrip())
        m = re.search(r"# \((\d+)\)", r)
        if m:
            actual, base = int(m.group(1)), ind
        elif r.strip() and ind < base:
            actual = None
        yield ("▶ " if r.strip() and actual in ls else "  ") + r

def codigo(f, fuentes):
    fuente = inspect.getsource(elegir(f, fuentes))
    return "\n".join(con_flechas(fuente, rango(f["linea"])))

def posiciones(t):
    """(x, y) de cada nodo: hojas en fila; un padre, al centro."""
    pos, hojas = {}, iter(range(99))
    def colocar(n):
        xs = [colocar(h) for h in t["hijos"].get(n, {}).values()]
        xs = xs or [next(hojas)]
        pos[n] = (sum(xs) / len(xs), -t["prof"][n])
        return pos[n][0]
    colocar("R")
    return pos


usar(ARBOL_T)
print("reglas listas · árbol T")

In [ ]:
# @title Visor: dibujo y deslizador (no hace falta leerla)
import matplotlib.pyplot as plt

try:
    from ipywidgets import (HBox, IntSlider, Play, VBox,
                            interactive_output, jslink)
    from IPython.display import display
    HAY_WIDGETS = True
except ImportError:                      # sin widgets: se imprime con un for
    HAY_WIDGETS = False

COLOR = {"pormirar": "#eeeeee", "pila": "#ffcf5c", "devuelto": "#8ecae6",
         "evaluado": "#a7d7a0", "podado": "#ffffff"}
MARCA = {"MAX": "^", "MIN": "v", "AZAR": "o", "HOJA": "s"}
LEYENDA = ("▲ MAX   ▼ MIN   ● azar   ■ hoja   ·   gris: sin generar   ·   "
           "amarillo: en la pila   ·   azul: devuelto\n"
           "verde: EVAL (una estimación)   ·   punteado con ?: podado   ·   "
           "«cota»: hubo corte; lo devuelto puede no ser su valor")


def tipo(t, n):
    if n in t["valor"]:
        return "HOJA"
    return "AZAR" if n in t["azar"] else ("MAX", "MIN")[t["prof"][n] % 2]


def nodo(ax, n, x, y, e, t):
    k, est = tipo(t, n), e["estado"]
    pila = est == "pila"
    ax.scatter(x, y, s=1300, marker=MARCA[k], c=COLOR[est], zorder=2,
               edgecolors="#d1495b" if pila else "#444",
               linewidths=3 if pila else 1.2,
               linestyle="--" if est == "podado" else "-")
    txt = "?" if est == "podado" else fmt(t["valor"].get(n, e["v"]))
    ax.text(x, y + {"^": -.06, "v": .06}.get(MARCA[k], 0), txt, zorder=3,
            ha="center", va="center", fontsize=10, weight="bold",
            color="#999" if est == "pormirar" else "#111")
    if k != "HOJA":
        ax.text(x, y + .4, n, ha="center", fontsize=9, color="#333")
    if e["alfa"] is not None and pila:
        ax.text(x + .3, y, f"({fmt(e['alfa'])}, {fmt(e['beta'])})",
                fontsize=9, color="#d1495b", va="center")
    if e["cota"]:
        ax.text(x, y - .42, "cota", ha="center", fontsize=8, c="#d1495b")


def dibujar(foto, t, ax, titulo=""):
    pos = posiciones(t)
    for p, hs in t["hijos"].items():
        for h in hs.values():
            ls = ":" if foto[h]["estado"] == "podado" else "-"
            ax.plot(*zip(pos[p], pos[h]), ls=ls, c="#888", zorder=1)
    for n, (x, y) in pos.items():
        nodo(ax, n, x, y, foto[n], t)
    ax.set_ylim(min(y for _, y in pos.values()) - .55, .8)
    ax.set_title(titulo, loc="left", fontsize=10)
    ax.axis("off")


def figura():
    fig, ax = plt.subplots(figsize=(9, 4.8))
    fig.text(.01, .01, LEYENDA, fontsize=8, color="#555")
    return fig, ax


def ver_arbol(t):
    vacio = dict(estado="pormirar", v=None, alfa=None, beta=None, cota=False)
    fig, ax = figura()
    dibujar({n: vacio for n in t["prof"]}, t, ax, "el árbol T")
    plt.show()


def mostrar(filas, fotos, fuentes, k):
    f = filas[k - 1]
    fig, ax = figura()
    titulo = (f"paso {k} de {len(filas)} · línea {f['linea'] or '—'} · "
              f"generados {len(f['hechos'])}/{len(f['t']['prof'])} · "
              f"pila {' › '.join(f['pila'])}")
    dibujar(fotos[k - 1], f["t"], ax, titulo)
    plt.show()
    print(renglon(f), "\n")
    print(codigo(f, fuentes) if f["linea"] else "(fila ✗: no corre nada)")


def ver(filas, *fuentes):
    """El deslizador de pasos; sin widgets, todos los pasos con un for."""
    fotos = fotos_de(filas)
    if not HAY_WIDGETS:
        tabla(filas)
        return mostrar(filas, fotos, fuentes, len(filas))
    barra = IntSlider(value=1, min=1, max=len(filas), description="paso")
    play = Play(value=1, min=1, max=len(filas), interval=1800)
    jslink((play, "value"), (barra, "value"))
    vista = interactive_output(
        lambda k: mostrar(filas, fotos, fuentes, k), {"k": barra})
    display(VBox([HBox([play, barra]), vista]))


def lupa(filas, k, *fuentes):            # un solo paso, fijo
    mostrar(filas, fotos_de(filas), fuentes, k)


print("visor listo · widgets:", HAY_WIDGETS)

## 2 · El árbol T

R es la raíz; mueve MAX. Sus jugadas `izq`, `centro` y `der` llevan a I, C
y D. De C salen `c1` y `c2` a C1 y C2. Los números son finales: su $U$.

In [ ]:
usar(ARBOL_T)
ver_arbol(JUEGO)
print("A(R) =", acciones("R"), "· A(C) =", acciones("C"))
print("T(C, c2) =", transicion("C", "c2"), "· Pl(C2) =", pl("C2"))

**Pregunta.** ¿Quién mueve en cada nivel? (▲ es MAX, ▼ es MIN.)

<details><summary>Respuesta</summary>

R mueve MAX; en I, C y D mueve MIN; en C1 y C2 otra vez MAX. Los turnos
alternan. Las hojas son finales: nadie mueve.

</details>

**Cómo leer el código.** Es el Python de la página. Dos añadidos:
`yield paso("12–13", locals())` toma una foto de las variables en esa fila
de la tabla, y `yield from` deja pasar las fotos de los hijos. En alfa-beta,
`yield from corte(...)` es la foto del corte y de cada hijo que ya no se
genera.

## 3 · Minimax

MAX se queda con el mayor $w$ de sus hijos; MIN, con el menor. La raíz no
devuelve un valor: devuelve **la jugada**.

In [ ]:
def decidir_minimax(s):         # (1)
    # (2) Nada visto: -inf pierde con todo.
    mejor_valor = -inf
    mejor_jugada = None
    yield paso("2", locals())
    for a in acciones(s):       # (3)
        # (4) Cuánto vale a si después los
        # dos juegan bien.
        w = yield from minimax(transicion(s, a))
        # (5) Solo si mejora estrictamente:
        # en un empate se queda la primera.
        if w > mejor_valor:
            mejor_valor = w
            mejor_jugada = a
        yield paso("4–5", locals())
    yield paso("6", locals())
    return mejor_jugada         # (6) jugada

In [ ]:
def minimax(s):                 # (7) da V(s)
    if es_final(s):             # (8) su U
        return utilidad(s)
    if pl(s) == "MAX":          # (9) mayor
        v = -inf                # (10)
        yield paso("10", locals())
        for a in acciones(s):   # (11)
            # (12) el hijo devuelve w
            w = yield from minimax(transicion(s, a))
            v = max(v, w)       # (13)
            yield paso("12–13", locals())
        return v                # (14)
    else:                       # (15) menor
        v = inf                 # (16)
        yield paso("16", locals())
        for a in acciones(s):   # (17)
            # (18) el hijo devuelve w
            w = yield from minimax(transicion(s, a))
            v = min(v, w)       # (19)
            yield paso("18–19", locals())
        return v                # (20)

In [ ]:
usar(ARBOL_T)
filas_mm, jugada = correr(decidir_minimax("R"))
print("jugada:", jugada, "· generados:", len(filas_mm[-1]["hechos"]), "de 14")
ver(filas_mm, decidir_minimax, minimax)

**Pruébalo.** Cambia la primera hoja de D, el 2, por un 9, y corre.

<details><summary>Qué pasa</summary>

D vale $\min(9, 12) = 9$, más que el 5 de C: sale `der`. Una sola hoja
cambió la jugada, porque minimax mira todo.

</details>

In [ ]:
prueba = ((3, 6), ((5, 2), (7, 8)), (2, 12))     # cambia un número
usar(prueba)
print("jugada:", correr(decidir_minimax("R"))[1])

## 4 · Cuando decide un dado: expectiminimax

Ahora I, C y D son **volados parejos**: cada hijo sale con probabilidad ½.
Un nodo de azar no elige: suma `Pr · w` de todos sus hijos. La tabla sigue
solo a los nodos de azar; C1 y C2 se valoran por dentro, sin filas.

In [ ]:
def decidir_expectiminimax(s):  # (1)
    # (2) Aún no ha visto ninguna jugada.
    mejor_valor = -inf
    mejor_jugada = None
    yield paso("2", locals())
    for a in acciones(s):       # (3)
        # (4) ¿Cuánto vale a, en promedio?
        w = yield from expectiminimax(transicion(s, a))
        # (5) Solo si mejora estrictamente.
        if w > mejor_valor:
            mejor_valor = w
            mejor_jugada = a
        yield paso("4–5", locals())
    yield paso("6", locals())
    return mejor_jugada         # (6) jugada

In [ ]:
def expectiminimax(s):          # (7)
    if es_final(s):             # (8) su U
        return utilidad(s)
    if pl(s) == "MAX":          # (9)
        v = -inf                # (10)
        for a in acciones(s):   # (11)
            w = yield from expectiminimax(  # (12)
                transicion(s, a))
            v = max(v, w)       # (13) mayor
        return v                # (14)
    elif pl(s) == "MIN":        # (15)
        v = inf                 # (16)
        for a in acciones(s):   # (17)
            w = yield from expectiminimax(  # (18)
                transicion(s, a))
            v = min(v, w)       # (19) menor
        return v                # (20)
    else:                       # (21) AZAR
        v = 0                   # (22) desde 0
        yield paso("22", locals())
        for a in acciones(s):   # (23) todos
            w = yield from expectiminimax(  # (24)
                transicion(s, a))
            v = v + pr(s, a) * w  # (25)
            yield paso("24–25", locals())
        return v                # (26)

In [ ]:
usar(ARBOL_T, azar=("I", "C", "D"))
filas_az, jugada = correr(decidir_expectiminimax("R"))
for f in filas_az:
    if f["evento"] == "raiz":
        print(f"{f['hijo']} vale {fmt(f['w'])}")
print("jugada:", jugada, "  (minimax decía centro)")
ver(filas_az, decidir_expectiminimax, expectiminimax)

**Pruébalo.** Cambia el 12 de D por un 4: ¿sigue ganando `der`?

<details><summary>Qué pasa</summary>

D vale $\tfrac12\cdot 2 + \tfrac12\cdot 4 = 3$, menos que $13/2$: gana
`centro`. Con azar cuenta cada hoja, no solo la peor.

</details>

In [ ]:
prueba = ((3, 6), ((5, 2), (7, 8)), (2, 12))     # cambia el 12
usar(prueba, azar=("I", "C", "D"))
print("jugada:", correr(decidir_expectiminimax("R"))[1])

## 5 · Alfa-beta

- **α**: lo que MAX ya tiene asegurado en el camino. Solo sube.
- **β**: lo que MIN ya tiene asegurado en el camino. Solo baja.

La regla: **MAX compara v ≥ β; MIN compara v ≤ α**. Cada uno mira el número
del rival, que heredó de arriba. Si se cumple, corta: el rival nunca
dejaría llegar la partida ahí.

In [ ]:
def decidir_alfa_beta(s):     # (1)
    # (2) alfa hace de mejor_valor
    alfa = -inf
    mejor_jugada = None
    yield paso("2", locals())
    for a in acciones(s):     # (3)
        # (4) pasa su alfa; beta: inf
        hijo = transicion(s, a)
        w = yield from alfa_beta(hijo, alfa, inf)
        # (5) estricto: una cota que empata
        # con alfa puede valer menos
        if w > alfa:
            alfa = w
            mejor_jugada = a
        yield paso("4–5", locals())
    yield paso("6", locals())
    return mejor_jugada       # (6)

In [ ]:
def alfa_beta(s, alfa, beta): # (7)
    if es_final(s):           # (8)
        return utilidad(s)
    if pl(s) == "MAX":        # (9)
        v = -inf              # (10)
        yield paso("10", locals())
        for a in acciones(s): # (11)
            # (12) hereda la ventana
            hijo = transicion(s, a)
            w = yield from alfa_beta(hijo, alfa, beta)
            v = max(v, w)     # (13)
            # (14) lee el del rival
            if v >= beta:
                yield from corte("12–14", locals())
                return v
            # (15) actualiza el suyo
            alfa = max(alfa, v)
            yield paso("12–15", locals())
        return v              # (16)
    else:                     # (17)
        v = inf               # (18)
        yield paso("18", locals())
        for a in acciones(s): # (19)
            # (20) hereda la ventana
            hijo = transicion(s, a)
            w = yield from alfa_beta(hijo, alfa, beta)
            v = min(v, w)     # (21)
            # (22) lee el del rival
            if v <= alfa:
                yield from corte("20–22", locals())
                return v
            # (23) actualiza el suyo
            beta = min(beta, v)
            yield paso("20–23", locals())
        return v              # (24)

In [ ]:
AB = (decidir_alfa_beta, alfa_beta)
usar(ARBOL_T1)                           # T sin el centro: 6 de 7
filas_t1, jugada = correr(decidir_alfa_beta("R"))
print("T1:", jugada, "·", len(filas_t1[-1]["hechos"]), "de 7 nodos")
ver(filas_t1, *AB)

Ahora T: 12 de 14. Minimax generaba los 14; la jugada es la misma.

In [ ]:
usar(ARBOL_T)
filas_ab, jugada = correr(decidir_alfa_beta("R"))
print("T:", jugada, "·", len(filas_ab[-1]["hechos"]), "de 14 nodos")
ver(filas_ab, *AB)

**Dos lupas.** En C2 (MAX): recibe 7, y $7 \ge \beta = 5$. C ya tiene 5 y
nunca escogería algo que vale 7 o más: la hoja 8 no se genera. C2
**devuelve 7 aunque vale 8**: es una cota, «al menos 7», no su valor.

En D (MIN): recibe 2, y $2 \le \alpha = 5$. R ya tiene 5 con `centro`, y
D vale a lo más 2: la hoja 12 no se genera.

In [ ]:
for corte_ in ("corte beta", "corte alfa"):
    k = next(f["n"] for f in filas_ab if f["evento"] == corte_)
    lupa(filas_ab, k, *AB)

**Pruébalo.** Cambia el 7 de C2 por un 4: ¿cuántos nodos genera ahora?

<details><summary>Qué pasa</summary>

13. C2 ya no corta en su primera hoja ($4 < \beta = 5$), así que genera
el 8; ahí $8 \ge 5$, pero ya no quedan hijos. C sigue valiendo 5 y la
jugada sigue siendo `centro`.

</details>

El ejercicio «Llena la traza con el orden invertido», de la página
*Alfa-beta como algoritmo*, resuélvelo primero a mano. Después revísalo con
`usar(ARBOL_T, al_reves=True)` y `ver(correr(decidir_alfa_beta("R"))[0], *AB)`.

In [ ]:
prueba = ((3, 6), ((5, 2), (7, 8)), (2, 12))     # cambia el 7
usar(prueba)
filas, jugada = correr(decidir_alfa_beta("R"))
print("jugada:", jugada, "·", len(filas[-1]["hechos"]), "nodos")

## 6 · Minimax con corte

Si el árbol no cabe, se mira solo `d` jugadas adelante; donde se acaba el
horizonte, el nodo vale `EVAL(s)`, una estimación. Desde aquí el árbol usa
la escala de la clase 3: la hoja 3 es un final con $U = 3/100$, y la línea
8 devuelve $100 \cdot U = 3$.

In [ ]:
def decidir_con_corte(s, d):    # (1)
    # (2) Aún no ha visto ninguna jugada.
    mejor_valor = -inf
    mejor_jugada = None
    yield paso("2", locals())
    for a in acciones(s):       # (3)
        # (4) Valora el hijo con una jugada
        # menos.
        w = yield from minimax_con_corte(
            transicion(s, a), d - 1)
        # (5) Solo si mejora estrictamente.
        if w > mejor_valor:
            mejor_valor = w
            mejor_jugada = a
        yield paso("4–5", locals())
    yield paso("6", locals())
    return mejor_jugada         # (6) jugada

In [ ]:
def minimax_con_corte(s, d):    # (7)
    # (8) Un final vale 100 veces su U.
    if es_final(s):
        return 100 * utilidad(s)
    # (9) Nodo de corte: se estima.
    if d == 0:
        return evaluar(s)
    if pl(s) == "MAX":          # (10)
        v = -inf                # (11)
        yield paso("11", locals())
        for a in acciones(s):   # (12)
            # (13) una jugada menos
            w = yield from minimax_con_corte(
                transicion(s, a), d - 1)
            v = max(v, w)       # (14) mayor
            yield paso("13–14", locals())
        return v                # (15)
    else:                       # (16) MIN
        v = inf                 # (17)
        yield paso("17", locals())
        for a in acciones(s):   # (18)
            # (19) una jugada menos
            w = yield from minimax_con_corte(
                transicion(s, a), d - 1)
            v = min(v, w)       # (20) menor
            yield paso("19–20", locals())
        return v                # (21)

In [ ]:
usar(ARBOL_T, cien=True)
for d in (1, 2, 3):
    filas, jugada = correr(decidir_con_corte("R", d))
    print(f"d = {d}: {jugada:<6} con {fmt(filas[-1]['v'])},",
          f"{len(filas[-1]['hechos'])} nodos generados")
filas_c2, _ = correr(decidir_con_corte("R", 2))
ver(filas_c2, decidir_con_corte, minimax_con_corte)

Con `d = 1` cae en una trampa: `der` parece valer 7, pero D vale 2. Con
`d = 2` ya ve la respuesta de MIN. En la figura, verde es `EVAL`.

**Pruébalo.** Con `d = 1`, sube `EVAL_T["C"]` a 8: ¿qué jugada sale?

<details><summary>Qué pasa</summary>

`centro`, con 8. Con el horizonte tan corto, la jugada la decide la
estimación, no el juego.

</details>

In [ ]:
EVAL_T["C"] = 4                          # cambia el 4
usar(ARBOL_T, cien=True)
print("d = 1:", correr(decidir_con_corte("R", 1))[1])
EVAL_T["C"] = 4                          # el de la página

## 7 · Profundización iterativa: jugar contra el reloj

No sabes cuánto tiempo queda. Busca con `d = 1`, luego `d = 2`, `d = 3`…
y entrega la jugada de **la última búsqueda completa**. La jugada anterior
va primero: α sube pronto y se poda más. Debajo de R corre el alfa-beta de
la sección 5, con `d`. Un 100 es $100\cdot U$ con $U = 1$: una victoria
segura (línea 9).

In [ ]:
# (14) El alfa-beta de la clase 2, con d.
def alfa_beta_con_corte(s, d, alfa, beta):
    if es_final(s):                # (15)
        return 100 * utilidad(s)
    if d == 0:                     # (16) EVAL
        return evaluar(s)
    if pl(s) == "MAX":             # (17)
        v = -inf                   # (18)
        yield paso("18", locals())
        for a in acciones(s):      # (19)
            w = yield from alfa_beta_con_corte(  # (20)
                transicion(s, a),
                d - 1, alfa, beta)
            v = max(v, w)          # (21)
            if v >= beta:          # (22) beta
                yield from corte("20–22", locals())
                return v
            alfa = max(alfa, v)    # (23)
            yield paso("20–23", locals())
        return v                   # (24)
    else:                          # (25)
        v = inf                    # (26)
        yield paso("26", locals())
        for a in acciones(s):      # (27)
            w = yield from alfa_beta_con_corte(  # (28)
                transicion(s, a),
                d - 1, alfa, beta)
            v = min(v, w)          # (29)
            if v <= alfa:          # (30) alfa
                yield from corte("28–30", locals())
                return v
            beta = min(beta, v)    # (31)
            yield paso("28–31", locals())
        return v                   # (32)

El reloj es una función, `queda_tiempo()`. El de la página alcanza justo
para tres búsquedas completas. Con `RELOJ["nodos"] = N`, el tiempo se acaba
al generar el nodo N; R lo revisa cada vez que vuelve un hijo (línea 8).

In [ ]:
RELOJ = {"nodos": None}                  # None: el reloj de la página

def queda_tiempo():
    if RELOJ["nodos"] is None:
        return CUENTA["terminadas"] < 3  # tres búsquedas completas
    return CUENTA["total"] < RELOJ["nodos"]

In [ ]:
def profundizacion_iterativa(s):   # (1)
    jugada = acciones(s)[0]        # (2)
    d = 1                          # (3)
    yield paso("2–3", locals())
    while queda_tiempo():          # (4)
        alfa = -inf                # (5)
        mejor_jugada = jugada
        yield paso("5", locals())
        # (6) La jugada anterior va primero.
        resto = [a for a in acciones(s)
                 if a != jugada]
        for a in [jugada] + resto:
            # (7) Le quedan d - 1 jugadas.
            w = yield from alfa_beta_con_corte(
                transicion(s, a),
                d - 1, alfa, inf)
            # (8) A medias: se descarta.
            if not queda_tiempo():
                return jugada
            # (9) Victoria asegurada.
            if w == 100:
                return a
            # (10) Mejor que lo visto.
            if w > alfa:
                alfa = w
                mejor_jugada = a
            yield paso("7–10", locals())
        jugada = mejor_jugada      # (11)
        yield paso("11–12", locals())
        d = d + 1                  # (12)
    yield paso("13", locals())
    return jugada                  # (13)

In [ ]:
usar(ARBOL_T, cien=True)
RELOJ["nodos"] = None
filas_pi, jugada = correr(profundizacion_iterativa("R"))
for f in filas_pi:
    if f["evento"] == "fin":
        print(f"d = {f['d']}: deja lista {f['jugada']:<6}",
              f"({len(f['hechos'])} nodos)")
print("se entrega:", jugada)
ver(filas_pi, profundizacion_iterativa, alfa_beta_con_corte)

**El reloj de N nodos.** ¿Qué jugada se entrega si el tiempo se acaba al
generar el nodo N, contando todas las búsquedas?

In [ ]:
antes = None
for n in range(1, 27):
    RELOJ["nodos"] = n
    jugada = correr(profundizacion_iterativa("R"))[1]
    if jugada != antes:
        print(f"desde N = {n:>2}: se entrega {jugada}")
        antes = jugada
RELOJ["nodos"] = None

**Pruébalo.** Con un reloj de 12 nodos se entrega `der`, aunque la
búsqueda con `d = 2` ya había visto que `centro` vale más. ¿Por qué?

<details><summary>Respuesta</summary>

Porque esa búsqueda no terminó. Una búsqueda a medias puede no haber visto
la respuesta que deshace su jugada; se entrega la de la última búsqueda
completa, la de `d = 1`.

</details>

## 8 · Cierre

| algoritmo | jugada | nodos generados |
|---|---|---|
| minimax | centro (5) | 14 de 14 |
| expectiminimax (I, C, D de azar) | der (7) | 14 de 14 |
| alfa-beta en T1 / en T | izq (3) / centro (5) | 6 de 7 / 12 de 14 |
| con corte, d = 1, 2, 3 | der, centro, centro | 4, 10, 14 |
| profundización iterativa | centro | 4 + 10 + 11 |

La celda de abajo vuelve a calcular toda la tabla.

In [ ]:
def resumen(nombre, algoritmo, arbol=ARBOL_T, **reglas):
    usar(arbol, **reglas)
    filas, jugada = correr(algoritmo())
    fines = [f for f in filas if f["evento"] == "fin"]
    nodos = " + ".join(str(len(f["hechos"])) for f in fines)
    print(f"{nombre:<14} {jugada:<7} {nodos} nodos")
resumen("minimax", lambda: decidir_minimax("R"))
resumen("azar", lambda: decidir_expectiminimax("R"), azar="ICD")
resumen("alfa-beta T1", lambda: decidir_alfa_beta("R"), ARBOL_T1)
resumen("alfa-beta T", lambda: decidir_alfa_beta("R"))
for d in (1, 2, 3):
    resumen(f"corte d = {d}", lambda: decidir_con_corte("R", d), cien=True)
resumen("iterativa", lambda: profundizacion_iterativa("R"), cien=True)
usar(ARBOL_T)

**Tres preguntas.**

1. ¿Por qué alfa-beta da la misma jugada que minimax aunque genere menos?
2. ¿Por qué expectiminimax elige `der` si minimax elige `centro`?
3. ¿Qué entrega la profundización iterativa si el reloj se acaba a mitad
   de `d = 3`?

<details><summary>Respuestas</summary>

1. Solo deja de mirar lo que no puede cambiar la decisión: un nodo podado
   está fuera de la ventana (α, β), y su padre ya tiene algo mejor.
2. Con azar, D ya no es «la peor hoja», sino el promedio: $(2 + 12)/2 = 7$.
3. `centro`, la jugada que dejó lista `d = 2`. La de `d = 3` está a medias.

</details>